<a href="https://colab.research.google.com/github/kuheli31/Fake-News-Detector/blob/main/Fake_News_New.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
#Importing libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from sklearn.metrics import confusion_matrix
from sklearn.metrics import classification_report
from sklearn.metrics import roc_auc_score
from sklearn.metrics import roc_curve
import re
import string
import requests
from requests.utils import quote
from urllib.parse import quote

Enter the news to verify if its real or fake

In [2]:
import spacy


nlp = spacy.load("en_core_web_sm")
news = input("Enter the news to verify if it is real or fake: ")
doc = nlp(news)
keywords = [token.text for token in doc if token.pos_ in ["PROPN", "NOUN"] and len(token.text) > 2]
seen = set()
keywords = [k for k in keywords if not (k.lower() in seen or seen.add(k.lower()))]
news_check = keywords[:4]
print("Extracted Keywords for Search:", " ".join(news_check))


Enter the news to verify if it is real or fake: India's win over Iran in the gold.
Extracted Keywords for Search: India win Iran gold


# **Data Collection**

**G-News API (Real News)**

In [3]:
#Real time news api (https://gnews.io/dashboard) -- 100 requests per day
news_api_key = 'd2b27a1e3f818564545508b6c744d24a'
#Endpoints are -> {search , top-headlines}
endpoint = 'search'
#Parameters are -> {The topic the user want to search}
question = " ".join(news_check)
parameters = f'q={quote(question)}&lang=en&max=5'
#Example (try this link to understand the url)-> https://gnews.io/api/v4/top-headlines?q=modi&lang=en&max=5&apikey=d2b27a1e3f818564545508b6c744d24a
news_api_url = f'https://gnews.io/api/v4/{endpoint}?{parameters}&apikey={news_api_key}'

response = requests.get(news_api_url)
data = response.json()

if 'articles' in data:
    for article in data['articles']:
        print(f"Published Time: {article['publishedAt']}")
        print(f"Title: {article['title']}")
        print(f"Description: {article['description']}")
        print(f"Image: {article['image']}")
        print(f"Content: {article['content']}")
        print(f"Source Name: {article['source']['name']}")
        print(f"Source URL: {article['source']['url']}")
        print(f"Source Country: {article['source']['country']}")
        print(f"URL: {article['url']}\n")
 # Extract article details into a list of dicts
    articles_data = []
    for article in data['articles']:
        articles_data.append({
            "Published Time": article.get('publishedAt'),
            "Title": article.get('title'),
            "Description": article.get('description'),
            "Image URL": article.get('image'),
            "Content": article.get('content'),
            "Source Name": article['source'].get('name'),
            "Source URL": article['source'].get('url'),
            "Source Country": article['source'].get('country'),
            "Article URL": article.get('url')
        })

    # Create DataFrame
    df = pd.DataFrame(articles_data)

    # Save as CSV
    df.to_csv('gnews.csv', index=False, encoding='utf-8')

    print("✅ Articles successfully saved to 'gnews.csv'")

else:
    print(data)

Published Time: 2026-09-27T08:47:17Z
Title: "I still can't believe it": Prachi Choudhary after clinching women's 400m bronze at Asian Games
Description: Prachi Choudhary claimed the bronze with a timing of 53.21 seconds. Bahrain's Salma Naser claimed the gold medal with a Games Record time of 49.20 seconds, while Iran's Zahra Zarei Manoujan took silver after clocking 52.38 seconds. India's Kiran finished fifth in 53.41 seconds. Following the medal win, Prachi described the moment as something she had been working for since a long time.
Image: https://www.tribuneindia.com/sortd-service/imaginary/v22-01/jpg/large/high?url=dGhldHJpYnVuZS1zb3J0ZC1wcm8tcHJvZC1zb3J0ZC9tZWRpYTBmZDNlMTEwLWJhNTAtMTFmMS1iN2M0LWQ5MzYzYTY1OTBjNy5qcGc=
Content: New Delhi [India], September 27 (ANI): India added to its medal tally at the Asian Games on Saturday as Prachi Choudhary claimed bronze in the final of the women’s 400m, delivering one of the standout performances of the night for the Indian continge... [267

**BlueSky API**

In [15]:
pip install atproto

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 623.2/623.2 kB 23.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 331.1/331.1 kB 16.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 286.0/286.0 kB 15.6 MB/s eta 0:00:00


In [22]:
# ============================================
# Bluesky API - Search Posts
# ============================================

import requests
import pandas as pd


# --------------------------------------------
# Search query
# --------------------------------------------

question = " ".join(news_check).strip()


# --------------------------------------------
# Bluesky public API endpoint
# --------------------------------------------

SEARCH_URL = "https://api.bsky.app/xrpc/app.bsky.feed.searchPosts"


# --------------------------------------------
# Check query
# --------------------------------------------

if not question:

    print("ERROR: Search query is required.")

else:

    try:

        # ------------------------------------
        # API parameters
        # ------------------------------------

        params = {
            "q": question,
            "limit": 10
        }


        # ------------------------------------
        # API request
        # ------------------------------------

        response = requests.get(
            SEARCH_URL,
            params=params,
            timeout=10
        )


        # ------------------------------------
        # Check response
        # ------------------------------------

        response.raise_for_status()

        data = response.json()

        posts = data.get("posts", [])


        # ------------------------------------
        # No posts found
        # ------------------------------------

        if not posts:

            print("\nNo Bluesky posts found for that query.")

            # Create empty DataFrame
            df_bluesky = pd.DataFrame()


        # ------------------------------------
        # Posts found
        # ------------------------------------

        else:

            print("\n--- Found Bluesky Posts ---\n")

            post_rows = []


            # --------------------------------
            # Process each post
            # --------------------------------

            for post in posts:

                # ----------------------------
                # Post information
                # ----------------------------

                record = post.get("record", {})
                author = post.get("author", {})

                post_text = record.get("text")
                created_at = record.get("createdAt")


                # ----------------------------
                # Author information
                # ----------------------------

                author_did = author.get("did")

                author_handle = author.get("handle")

                author_display_name = author.get(
                    "displayName"
                )

                author_avatar = author.get(
                    "avatar"
                )


                # ----------------------------
                # Post URI
                # ----------------------------

                post_uri = post.get("uri")


                # ----------------------------
                # Create post URL
                # ----------------------------

                if author_handle and post_uri:

                    post_id = post_uri.split("/")[-1]

                    post_url = (
                        f"https://bsky.app/profile/"
                        f"{author_handle}/post/{post_id}"
                    )

                else:

                    post_url = None


                # ----------------------------
                # Engagement information
                # ----------------------------

                likes = post.get(
                    "likeCount",
                    0
                )

                reposts = post.get(
                    "repostCount",
                    0
                )

                replies = post.get(
                    "replyCount",
                    0
                )

                quotes = post.get(
                    "quoteCount",
                    0
                )


                # ----------------------------
                # Author URL
                # ----------------------------

                if author_handle:

                    author_url = (
                        f"https://bsky.app/profile/"
                        f"{author_handle}"
                    )

                else:

                    author_url = None


                # ----------------------------
                # Print information
                # ----------------------------

                print(
                    f"Published Time: {created_at}"
                )

                print(
                    f"Post Text: {post_text}"
                )

                print(
                    f"Author: "
                    f"{author_display_name} "
                    f"(@{author_handle})"
                )

                print(
                    f"Author DID: {author_did}"
                )

                print(
                    f"Author Profile Image: "
                    f"{author_avatar}"
                )

                print(
                    f"Author URL: {author_url}"
                )

                print(
                    f"Post URL: {post_url}"
                )

                print(
                    f"Likes: {likes}"
                )

                print(
                    f"Reposts: {reposts}"
                )

                print(
                    f"Replies: {replies}"
                )

                print(
                    f"Quotes: {quotes}"
                )

                print("\n")


                # ----------------------------
                # Store post information
                # ----------------------------

                post_rows.append({

                    "Published Time": created_at,

                    "Post Text": post_text,

                    "Author Name":
                        author_display_name,

                    "Author Handle":
                        author_handle,

                    "Author DID":
                        author_did,

                    "Author Profile Image":
                        author_avatar,

                    "Author URL":
                        author_url,

                    "Post URL":
                        post_url,

                    "Likes":
                        likes,

                    "Reposts":
                        reposts,

                    "Replies":
                        replies,

                    "Quotes":
                        quotes
                })


            # --------------------------------
            # Create DataFrame
            # --------------------------------

            df_bluesky = pd.DataFrame(
                post_rows
            )


            # --------------------------------
            # Save DataFrame to CSV
            # --------------------------------

            df_bluesky.to_csv(
                "bluesky_posts.csv",
                index=False,
                encoding="utf-8"
            )


            print(
                "✅ Bluesky posts successfully "
                "saved to 'bluesky_posts.csv'"
            )


            # --------------------------------
            # Display DataFrame
            # --------------------------------

            print("\n--- Bluesky DataFrame ---\n")

            display(df_bluesky)


    # ----------------------------------------
    # HTTP errors
    # ----------------------------------------

    except requests.exceptions.HTTPError as e:

        print(
            "\n--- BLUESKY API ERROR ---"
        )

        print(
            f"HTTP Error: {e}"
        )


    # ----------------------------------------
    # Connection errors
    # ----------------------------------------

    except requests.exceptions.RequestException as e:

        print(
            "\n--- CONNECTION ERROR ---"
        )

        print(
            f"Error details: {e}"
        )


    # ----------------------------------------
    # Other errors
    # ----------------------------------------

    except Exception as e:

        print(
            "\nAn unexpected error occurred."
        )

        print(
            f"Error details: {e}"
        )


--- Found Bluesky Posts ---

Published Time: 2026-09-26T17:10:25.000Z
Post Text: 
Author: Mashable India [Unofficial] (@in.mashable.com.web.brid.gy)
Author DID: did:plc:tkzj743ruh5itf2pov2q5txv
Author Profile Image: https://cdn.bsky.app/img/avatar/plain/did:plc:tkzj743ruh5itf2pov2q5txv/bafkreiaajie5hdpgbly5yopoisnlwz74khfw5sozjljda3dfhahzta7z4i
Author URL: https://bsky.app/profile/in.mashable.com.web.brid.gy
Post URL: https://bsky.app/profile/in.mashable.com.web.brid.gy/post/3mwgxvjs6y4z2
Likes: 0
Reposts: 0
Replies: 0
Quotes: 0


Published Time: 2026-09-26T16:07:43Z
Post Text: Asian Games 2026 Day 7: India win seven medals as kabaddi teams deliver gold double in action-packed Saturday | Asian Games 2026 News

L-R: Joshna Chinappa, Prachi Choudhary and Tajinderpal Singh Toor NEW DELHI: India enjoyed one of its most productive days at the Asian Games 2026 on Saturday,…
Author: HFG INSIDER (@health897.bsky.social)
Author DID: did:plc:euogifoh4mj52xvkbqcfqtti
Author Profile Image: https:

,Published Time,Post Text,Author Name,Author Handle,Author DID,Author Profile Image,Author URL,Post URL,Likes,Reposts,Replies,Quotes
0,2026-09-26T17:10:25.000Z,,Mashable India [Unofficial],in.mashable.com.web.brid.gy,did:plc:tkzj743ruh5itf2pov2q5txv,https://cdn.bsky.app/img/avatar/plain/did:plc:...,https://bsky.app/profile/in.mashable.com.web.b...,https://bsky.app/profile/in.mashable.com.web.b...,0,0,0,0
1,2026-09-26T16:07:43Z,Asian Games 2026 Day 7: India win seven medals...,HFG INSIDER,health897.bsky.social,did:plc:euogifoh4mj52xvkbqcfqtti,https://cdn.bsky.app/img/avatar/plain/did:plc:...,https://bsky.app/profile/health897.bsky.social,https://bsky.app/profile/health897.bsky.social...,0,0,0,0
2,2026-09-26T10:55:45.726Z,India beat Iran to win ninth Gold medal in men...,,newsinc24.bsky.social,did:plc:6qomd7wrtbnrjq5p3cww7fuc,https://cdn.bsky.app/img/avatar/plain/did:plc:...,https://bsky.app/profile/newsinc24.bsky.social,https://bsky.app/profile/newsinc24.bsky.social...,0,0,0,0
3,2026-09-26T09:36:23.000Z,,Mashable India [Unofficial],in.mashable.com.web.brid.gy,did:plc:tkzj743ruh5itf2pov2q5txv,https://cdn.bsky.app/img/avatar/plain/did:plc:...,https://bsky.app/profile/in.mashable.com.web.b...,https://bsky.app/profile/in.mashable.com.web.b...,0,0,0,0
4,2026-09-25T18:59:24Z,Asian Games 2026 Kabaddi: India Storm Into Men...,,newsarea.bsky.social,did:plc:kqj4vuyjvmtnptvvdt2xijtk,https://cdn.bsky.app/img/avatar/plain/did:plc:...,https://bsky.app/profile/newsarea.bsky.social,https://bsky.app/profile/newsarea.bsky.social/...,0,0,0,0
5,2026-09-25T07:18:03Z,Asian Games 2026 Day 6: Suruchi Singh and Kama...,,navroze-india.bsky.social,did:plc:nndhrtds5t6ipm4zdvov6php,https://cdn.bsky.app/img/avatar/plain/did:plc:...,https://bsky.app/profile/navroze-india.bsky.so...,https://bsky.app/profile/navroze-india.bsky.so...,0,0,0,0
6,2026-06-22T21:24:44.002Z,🗞️ Friday Finance #35 is live! This week: bill...,sam's friday finance weekly-ish,fridayfinance.bsky.social,did:plc:mb3dgo74f2yssbnccjfh4g6z,https://cdn.bsky.app/img/avatar/plain/did:plc:...,https://bsky.app/profile/fridayfinance.bsky.so...,https://bsky.app/profile/fridayfinance.bsky.so...,0,0,1,0
7,2025-10-23T14:57:32.319Z,India are the first girls' champs in kabaddi's...,Archive,archive041716.bsky.social,did:plc:de2ulpepjaul36mazt33vvdf,https://cdn.bsky.app/img/avatar/plain/did:plc:...,https://bsky.app/profile/archive041716.bsky.so...,https://bsky.app/profile/archive041716.bsky.so...,0,0,0,0
8,2025-08-25T09:01:05.356020+00:00,"Feed: ""Neptune Prime""\nBy: Maryam Sulaiman Abu...",Longtail News,longtail-news.bsky.social,did:plc:o6ggjvnj4ze3mnrpnv5oravg,https://cdn.bsky.app/img/avatar/plain/did:plc:...,https://bsky.app/profile/longtail-news.bsky.so...,https://bsky.app/profile/longtail-news.bsky.so...,0,0,0,0


**Reddit API**

In [20]:
!pip install praw

In [23]:
# Importing necessary libraries for Reddit API interaction
import praw
import datetime  # To format the timestamp from Reddit
import pandas as pd

# --- API Configuration ---
REDDIT_CLIENT_ID = 'utnSmiRRrv9ZDIlJ9LveMg'
REDDIT_CLIENT_SECRET = 'jzP9XRolmRsu3CNiDZmKF0IDDGcXpA'
REDDIT_USER_AGENT = 'python:moonsearcher:v1.0 (by /u/Dependent_Car_5879)'

# --- Search Parameters ---
try:
    question = " ".join(news_check).strip()
    num_results = 100  # Automatically take up to 100 results
except EOFError:
    print("\nNo input provided for search query. Exiting.")
    exit()

# --- Main Execution Block ---
if not REDDIT_CLIENT_ID or not REDDIT_CLIENT_SECRET or not question:
    print("ERROR: Client ID, Client Secret, and a search query are required.")
else:
    try:
        reddit = praw.Reddit(
            client_id=REDDIT_CLIENT_ID,
            client_secret=REDDIT_CLIENT_SECRET,
            user_agent=REDDIT_USER_AGENT,
        )

        search_results = reddit.subreddit("all").search(
            query=question,
            sort='new',
            limit=num_results
        )

        submissions = list(search_results)

        if not submissions:
            print("\nNo recent posts found for that query.")
        else:
            print(f"\n--- Found {len(submissions)} Posts ---\n")


            reddit_rows = []

            for submission in submissions:
                author_name = submission.author.name if submission.author else "[deleted]"
                post_url = f"https://www.reddit.com{submission.permalink}"
                author_url = f"https://www.reddit.com/user/{author_name}" if author_name != "[deleted]" else "N/A"
                published_time = datetime.datetime.fromtimestamp(submission.created_utc)


                print(f"Published Time: {published_time.strftime('%Y-%m-%d %H:%M:%S')}")
                print(f"Post Title: {submission.title}")
                print(f"Subreddit: r/{submission.subreddit.display_name}")
                print(f"Score: {submission.score} (Upvote Ratio: {submission.upvote_ratio * 100:.0f}%)")
                print(f"Author: u/{author_name}")
                print(f"Author URL: {author_url}")
                print(f"Post URL: {post_url}")

                post_text = submission.selftext.replace(chr(10), ' ') if submission.selftext else None


                image_url = None
                if submission.url.endswith(('.jpg', '.jpeg', '.png', '.gif')):
                    image_url = submission.url
                elif hasattr(submission, 'preview'):
                    try:
                        image_url = submission.preview['images'][0]['source']['url']
                    except (KeyError, IndexError):
                        pass

                if image_url:
                    print(f"Image URL: {image_url}")
                else:
                    print("Image: None")

                print("\n" + "-" * 60 + "\n")

                reddit_rows.append({
                    "Published Time": published_time.strftime('%Y-%m-%d %H:%M:%S'),
                    "Post Title": submission.title,
                    "Subreddit": submission.subreddit.display_name,
                    "Score": submission.score,
                    "Upvote Ratio (%)": f"{submission.upvote_ratio * 100:.0f}",
                    "Author": author_name,
                    "Author URL": author_url,
                    "Post URL": post_url,
                    "Post Text": post_text,
                    "Image URL": image_url
                })


            df_reddit = pd.DataFrame(reddit_rows)
            df_reddit.to_csv('reddit.csv', index=False, encoding='utf-8')

            print("✅ Reddit posts successfully saved to 'reddit.csv'")

    except praw.exceptions.PRAWException as e:
        print("\n--- API ERROR ---")
        print("An error occurred with the Reddit API. This could be due to invalid credentials or a connection problem.")
        print(f"Error details: {e}")
    except Exception as e:
        print(f"\nAn unexpected error occurred.")
        print(f"Error details: {e}")

        # --- Display DataFrame ---
    print("\n--- Reddit DataFrame ---\n")
    display(df_reddit)


It is strongly recommended to use Async PRAW: https://asyncpraw.readthedocs.io.
See https://praw.readthedocs.io/en/latest/getting_started/multiple_instances.html#discord-bots-and-asynchronous-environments for more info.

It is strongly recommended to use Async PRAW: https://asyncpraw.readthedocs.io.
See https://praw.readthedocs.io/en/latest/getting_started/multiple_instances.html#discord-bots-and-asynchronous-environments for more info.

It is strongly recommended to use Async PRAW: https://asyncpraw.readthedocs.io.
See https://praw.readthedocs.io/en/latest/getting_started/multiple_instances.html#discord-bots-and-asynchronous-environments for more info.

It is strongly recommended to use Async PRAW: https://asyncpraw.readthedocs.io.
See https://praw.readthedocs.io/en/latest/getting_started/multiple_instances.html#discord-bots-and-asynchronous-environments for more info.




--- Found 100 Posts ---

Published Time: 2026-09-28 07:58:05
Post Title: 📊 NordFX: Quarter-End Test - JOLTS, PCE, USD and Gold Into the Q3 Close
Subreddit: r/NordFX
Score: 1 (Upvote Ratio: 100%)
Author: u/Marketing_NordFX
Author URL: https://www.reddit.com/user/Marketing_NordFX
Post URL: https://www.reddit.com/r/NordFX/comments/1ws8i61/nordfx_quarterend_test_jolts_pce_usd_and_gold/
Image URL: https://external-preview.redd.it/ODNnbmoweGV3N3NoMTHlsiULYxYa5l1ejbAt3UwD4WA1z-qfgpIhryOGRvXi.png?format=pjpg&auto=webp&s=bf029dd0710999b3e53ea16819a1ab27a2d9dc23

------------------------------------------------------------

Published Time: 2026-09-28 07:50:40
Post Title: After using a GTX 1060 mobile since it came I finally got an RTX5070 PC. I played everything though.
Subreddit: r/gamesuggestions
Score: 2 (Upvote Ratio: 100%)
Author: u/AtmosphereBlue
Author URL: https://www.reddit.com/user/AtmosphereBlue
Post URL: https://www.reddit.com/r/gamesuggestions/comments/1ws8e56/after_using_a_gtx_106

It is strongly recommended to use Async PRAW: https://asyncpraw.readthedocs.io.
See https://praw.readthedocs.io/en/latest/getting_started/multiple_instances.html#discord-bots-and-asynchronous-environments for more info.

It is strongly recommended to use Async PRAW: https://asyncpraw.readthedocs.io.
See https://praw.readthedocs.io/en/latest/getting_started/multiple_instances.html#discord-bots-and-asynchronous-environments for more info.



Image: None

------------------------------------------------------------

Published Time: 2026-09-28 07:31:05
Post Title: This is why I can't get a better winrate on Mathilda... Mid lane is the most toxic role by far.
Subreddit: r/MobileLegendsGame
Score: 1 (Upvote Ratio: 100%)
Author: u/Dependent_Wafer3866
Author URL: https://www.reddit.com/user/Dependent_Wafer3866
Post URL: https://www.reddit.com/r/MobileLegendsGame/comments/1ws834g/this_is_why_i_cant_get_a_better_winrate_on/
Image: None

------------------------------------------------------------

Published Time: 2026-09-28 07:26:56
Post Title: 📊 12:00 PM GIFT NIFTY Update
Subreddit: r/GIFTNiftyLive
Score: 1 (Upvote Ratio: 100%)
Author: u/giftniftylive
Author URL: https://www.reddit.com/user/giftniftylive
Post URL: https://www.reddit.com/r/GIFTNiftyLive/comments/1ws80th/1200_pm_gift_nifty_update/
Image URL: https://i.redd.it/j2p649ysq7sh1.jpeg

------------------------------------------------------------

Published Time: 2026-09-2

It is strongly recommended to use Async PRAW: https://asyncpraw.readthedocs.io.
See https://praw.readthedocs.io/en/latest/getting_started/multiple_instances.html#discord-bots-and-asynchronous-environments for more info.

It is strongly recommended to use Async PRAW: https://asyncpraw.readthedocs.io.
See https://praw.readthedocs.io/en/latest/getting_started/multiple_instances.html#discord-bots-and-asynchronous-environments for more info.



...

It is strongly recommended to use Async PRAW: https://asyncpraw.readthedocs.io.
See https://praw.readthedocs.io/en/latest/getting_started/multiple_instances.html#discord-bots-and-asynchronous-environments for more info.

It is strongly recommended to use Async PRAW: https://asyncpraw.readthedocs.io.
See https://praw.readthedocs.io/en/latest/getting_started/multiple_instances.html#discord-bots-and-asynchronous-environments for more info.



Image: None

------------------------------------------------------------

Published Time: 2026-09-28 02:49:53
Post Title: “After weeks, I’m finally about to beat my Gold Stake Win Streak Record! This run was hard but we adapted, nothing is going to go wro-“
Subreddit: r/balatro
Score: 101 (Upvote Ratio: 99%)
Author: u/Bekenshi
Author URL: https://www.reddit.com/user/Bekenshi
Post URL: https://www.reddit.com/r/balatro/comments/1ws32h7/after_weeks_im_finally_about_to_beat_my_gold/
Image URL: https://external-preview.redd.it/bW14eDR4cWZkNnNoMUi7s940rG1r2L5fpuHKWvd7wdBZLAAgTav24HRPIgN3.jpeg?format=pjpg&auto=webp&s=a6283aaf152e8aaf87ad6e5c5b8150605c511594

------------------------------------------------------------

Published Time: 2026-09-28 02:43:54
Post Title: Event: Cursed Clear (Sep. 29-30, 2026) - Disney Emoji Blitz
Subreddit: r/disneyemojiblitz
Score: 23 (Upvote Ratio: 100%)
Author: u/EmojiBlitzerGurl
Author URL: https://www.reddit.com/user/EmojiBlitzerGurl
Post URL: https://www.re

It is strongly recommended to use Async PRAW: https://asyncpraw.readthedocs.io.
See https://praw.readthedocs.io/en/latest/getting_started/multiple_instances.html#discord-bots-and-asynchronous-environments for more info.



Image: None

------------------------------------------------------------

Published Time: 2026-09-28 02:31:46
Post Title: Best custom black volleyball jerseys websites
Subreddit: r/Customjerseyspro
Score: 1 (Upvote Ratio: 100%)
Author: u/fashioncustom
Author URL: https://www.reddit.com/user/fashioncustom
Post URL: https://www.reddit.com/r/Customjerseyspro/comments/1ws2pyy/best_custom_black_volleyball_jerseys_websites/
Image URL: https://external-preview.redd.it/EBGZJNTa6m5jk_ykg5YQDcz48AcaG24_Czn_FJotGQU.jpeg?auto=webp&s=85414a0813f4d2a9177b3311ea9ee38285c45d84

------------------------------------------------------------

Published Time: 2026-09-28 02:30:47
Post Title: India Ranks 10th at World Skills Shanghai 2026, Wins 6 Silver Medals and 20 Medal for Excellence.
Subreddit: r/BharatiyaJanataParty
Score: 2 (Upvote Ratio: 100%)
Author: u/CalmlyPassionate
Author URL: https://www.reddit.com/user/CalmlyPassionate
Post URL: https://www.reddit.com/r/BharatiyaJanataParty/comments/1ws2pap/i

,Published Time,Post Title,Subreddit,Score,Upvote Ratio (%),Author,Author URL,Post URL,Post Text,Image URL
0,2026-09-28 07:58:05,"📊 NordFX: Quarter-End Test - JOLTS, PCE, USD a...",NordFX,1,100,Marketing_NordFX,https://www.reddit.com/user/Marketing_NordFX,https://www.reddit.com/r/NordFX/comments/1ws8i...,🔥 **Can gold regain its footing as the dollar ...,https://external-preview.redd.it/ODNnbmoweGV3N...
1,2026-09-28 07:50:40,After using a GTX 1060 mobile since it came I ...,gamesuggestions,2,100,AtmosphereBlue,https://www.reddit.com/user/AtmosphereBlue,https://www.reddit.com/r/gamesuggestions/comme...,I finally have a new PC which has the RTX 5070...,None
2,2026-09-28 07:46:39,[Business] - Gold price prediction: What are t...,MetalsOnReddit,1,100,Then_Marionberry_259,https://www.reddit.com/user/Then_Marionberry_259,https://www.reddit.com/r/MetalsOnReddit/commen...,None,https://external-preview.redd.it/iHSGbIRF4mQhB...
3,2026-09-28 07:44:10,Mahanirvana Tantra Chapter 9 Part 6,TantraMarg,2,100,Straight-Cable433,https://www.reddit.com/user/Straight-Cable433,https://www.reddit.com/r/TantraMarg/comments/1...,MAHANIRVANA TANTRA CHAPTER IX: THE TEN SAMSKA...,https://i.redd.it/um9qciiyt7sh1.jpeg
4,2026-09-28 07:37:21,Got scammed by Zevana - “India’s most trusted ...,indianbeautyyappers,2,100,Effective_Lawyer_647,https://www.reddit.com/user/Effective_Lawyer_647,https://www.reddit.com/r/indianbeautyyappers/c...,Zevana claims to be India’s most trusted desti...,None
...,...,...,...,...,...,...,...,...,...,...
95,2026-09-28 02:36:55,rajesh kazhipurath,dalalstreetbets,1,100,ykar648,https://www.reddit.com/user/ykar648,https://www.reddit.com/r/dalalstreetbets/comme...,US long-term borrowing costs are flashing a wa...,https://external-preview.redd.it/7r2ejLzIFphhJ...
96,2026-09-28 02:34:37,Some winners of WorldSkills 2026 held in Shanghai,aznidentity,24,100,ding_nei_go_fei,https://www.reddit.com/user/ding_nei_go_fei,https://www.reddit.com/r/aznidentity/comments/...,WorldSkills now in it's 48th year is a competi...,None
97,2026-09-28 02:32:02,"CHANCE Indian Intl. applying Dartmouth ED, Eng...",chanceme,2,100,CatchPsychological95,https://www.reddit.com/user/CatchPsychological95,https://www.reddit.com/r/chanceme/comments/1ws...,"**Demographics:** * Indian, male, internation...",None
98,2026-09-28 02:31:46,Best custom black volleyball jerseys websites,Customjerseyspro,1,100,fashioncustom,https://www.reddit.com/user/fashioncustom,https://www.reddit.com/r/Customjerseyspro/comm...,Looking to customize a black volleyball jersey...,https://external-preview.redd.it/EBGZJNTa6m5jk...


# **Data Preprocessing**

**Real News**

In [7]:
df

,Published Time,Title,Description,Image URL,Content,Source Name,Source URL,Source Country,Article URL
0,2026-09-27T08:47:17Z,"""I still can't believe it"": Prachi Choudhary a...",Prachi Choudhary claimed the bronze with a tim...,https://www.tribuneindia.com/sortd-service/ima...,"New Delhi [India], September 27 (ANI): India a...",The Tribune,https://www.tribuneindia.com,in,https://www.tribuneindia.com/news/sports/i-sti...
1,2026-09-27T08:45:34Z,"""I still can't believe it"": Prachi Choudhary a...",Prachi Choudhary claimed the bronze with a tim...,https://d3lzcn6mbbadaf.cloudfront.net/media/de...,"New Delhi [India], September 27 (ANI): India a...",ANI (Asian News International),https://www.aninews.in,in,https://www.aninews.in/news/sports/others/i-st...
2,2026-09-26T17:10:25Z,India Beat Iran In Both Kabaddi Finals To Win ...,India completed a kabaddi golden double at the...,https://sm.mashable.com/mashable_in/seo/defaul...,India completed a golden double in kabaddi at ...,Mashable India,https://in.mashable.com,in,https://in.mashable.com/sports/114326/india-be...
3,2026-09-26T13:51:00Z,India maintain choke-hold on Asiad Kabaddi gol...,India's kabaddi teams delivered stellar perfor...,"https://img.etimg.com/thumb/msid-134505784,wid...",Synopsis\nIndia's kabaddi teams delivered stel...,The Economic Times,https://economictimes.indiatimes.com,in,https://economictimes.indiatimes.com/news/spor...
4,2026-09-26T12:52:34Z,Prachi Choudhary Win Bronze In Women's 400m Fo...,Salwa Naser of Bahrain bagged the gold medal w...,https://images.news18.com/ibnlive/uploads/2026...,"Last Updated:September 26, 2026, 18:27 IST\nSa...",News18,https://www.news18.com,in,https://www.news18.com/sports/other-sports/pra...


**BlueSky News**

In [24]:
df_bluesky

,Published Time,Post Text,Author Name,Author Handle,Author DID,Author Profile Image,Author URL,Post URL,Likes,Reposts,Replies,Quotes
0,2026-09-26T17:10:25.000Z,,Mashable India [Unofficial],in.mashable.com.web.brid.gy,did:plc:tkzj743ruh5itf2pov2q5txv,https://cdn.bsky.app/img/avatar/plain/did:plc:...,https://bsky.app/profile/in.mashable.com.web.b...,https://bsky.app/profile/in.mashable.com.web.b...,0,0,0,0
1,2026-09-26T16:07:43Z,Asian Games 2026 Day 7: India win seven medals...,HFG INSIDER,health897.bsky.social,did:plc:euogifoh4mj52xvkbqcfqtti,https://cdn.bsky.app/img/avatar/plain/did:plc:...,https://bsky.app/profile/health897.bsky.social,https://bsky.app/profile/health897.bsky.social...,0,0,0,0
2,2026-09-26T10:55:45.726Z,India beat Iran to win ninth Gold medal in men...,,newsinc24.bsky.social,did:plc:6qomd7wrtbnrjq5p3cww7fuc,https://cdn.bsky.app/img/avatar/plain/did:plc:...,https://bsky.app/profile/newsinc24.bsky.social,https://bsky.app/profile/newsinc24.bsky.social...,0,0,0,0
3,2026-09-26T09:36:23.000Z,,Mashable India [Unofficial],in.mashable.com.web.brid.gy,did:plc:tkzj743ruh5itf2pov2q5txv,https://cdn.bsky.app/img/avatar/plain/did:plc:...,https://bsky.app/profile/in.mashable.com.web.b...,https://bsky.app/profile/in.mashable.com.web.b...,0,0,0,0
4,2026-09-25T18:59:24Z,Asian Games 2026 Kabaddi: India Storm Into Men...,,newsarea.bsky.social,did:plc:kqj4vuyjvmtnptvvdt2xijtk,https://cdn.bsky.app/img/avatar/plain/did:plc:...,https://bsky.app/profile/newsarea.bsky.social,https://bsky.app/profile/newsarea.bsky.social/...,0,0,0,0
5,2026-09-25T07:18:03Z,Asian Games 2026 Day 6: Suruchi Singh and Kama...,,navroze-india.bsky.social,did:plc:nndhrtds5t6ipm4zdvov6php,https://cdn.bsky.app/img/avatar/plain/did:plc:...,https://bsky.app/profile/navroze-india.bsky.so...,https://bsky.app/profile/navroze-india.bsky.so...,0,0,0,0
6,2026-06-22T21:24:44.002Z,🗞️ Friday Finance #35 is live! This week: bill...,sam's friday finance weekly-ish,fridayfinance.bsky.social,did:plc:mb3dgo74f2yssbnccjfh4g6z,https://cdn.bsky.app/img/avatar/plain/did:plc:...,https://bsky.app/profile/fridayfinance.bsky.so...,https://bsky.app/profile/fridayfinance.bsky.so...,0,0,1,0
7,2025-10-23T14:57:32.319Z,India are the first girls' champs in kabaddi's...,Archive,archive041716.bsky.social,did:plc:de2ulpepjaul36mazt33vvdf,https://cdn.bsky.app/img/avatar/plain/did:plc:...,https://bsky.app/profile/archive041716.bsky.so...,https://bsky.app/profile/archive041716.bsky.so...,0,0,0,0
8,2025-08-25T09:01:05.356020+00:00,"Feed: ""Neptune Prime""\nBy: Maryam Sulaiman Abu...",Longtail News,longtail-news.bsky.social,did:plc:o6ggjvnj4ze3mnrpnv5oravg,https://cdn.bsky.app/img/avatar/plain/did:plc:...,https://bsky.app/profile/longtail-news.bsky.so...,https://bsky.app/profile/longtail-news.bsky.so...,0,0,0,0


**Reddit News**

In [25]:
df_reddit

,Published Time,Post Title,Subreddit,Score,Upvote Ratio (%),Author,Author URL,Post URL,Post Text,Image URL
0,2026-09-28 07:58:05,"📊 NordFX: Quarter-End Test - JOLTS, PCE, USD a...",NordFX,1,100,Marketing_NordFX,https://www.reddit.com/user/Marketing_NordFX,https://www.reddit.com/r/NordFX/comments/1ws8i...,🔥 **Can gold regain its footing as the dollar ...,https://external-preview.redd.it/ODNnbmoweGV3N...
1,2026-09-28 07:50:40,After using a GTX 1060 mobile since it came I ...,gamesuggestions,2,100,AtmosphereBlue,https://www.reddit.com/user/AtmosphereBlue,https://www.reddit.com/r/gamesuggestions/comme...,I finally have a new PC which has the RTX 5070...,None
2,2026-09-28 07:46:39,[Business] - Gold price prediction: What are t...,MetalsOnReddit,1,100,Then_Marionberry_259,https://www.reddit.com/user/Then_Marionberry_259,https://www.reddit.com/r/MetalsOnReddit/commen...,None,https://external-preview.redd.it/iHSGbIRF4mQhB...
3,2026-09-28 07:44:10,Mahanirvana Tantra Chapter 9 Part 6,TantraMarg,2,100,Straight-Cable433,https://www.reddit.com/user/Straight-Cable433,https://www.reddit.com/r/TantraMarg/comments/1...,MAHANIRVANA TANTRA CHAPTER IX: THE TEN SAMSKA...,https://i.redd.it/um9qciiyt7sh1.jpeg
4,2026-09-28 07:37:21,Got scammed by Zevana - “India’s most trusted ...,indianbeautyyappers,2,100,Effective_Lawyer_647,https://www.reddit.com/user/Effective_Lawyer_647,https://www.reddit.com/r/indianbeautyyappers/c...,Zevana claims to be India’s most trusted desti...,None
...,...,...,...,...,...,...,...,...,...,...
95,2026-09-28 02:36:55,rajesh kazhipurath,dalalstreetbets,1,100,ykar648,https://www.reddit.com/user/ykar648,https://www.reddit.com/r/dalalstreetbets/comme...,US long-term borrowing costs are flashing a wa...,https://external-preview.redd.it/7r2ejLzIFphhJ...
96,2026-09-28 02:34:37,Some winners of WorldSkills 2026 held in Shanghai,aznidentity,24,100,ding_nei_go_fei,https://www.reddit.com/user/ding_nei_go_fei,https://www.reddit.com/r/aznidentity/comments/...,WorldSkills now in it's 48th year is a competi...,None
97,2026-09-28 02:32:02,"CHANCE Indian Intl. applying Dartmouth ED, Eng...",chanceme,2,100,CatchPsychological95,https://www.reddit.com/user/CatchPsychological95,https://www.reddit.com/r/chanceme/comments/1ws...,"**Demographics:** * Indian, male, internation...",None
98,2026-09-28 02:31:46,Best custom black volleyball jerseys websites,Customjerseyspro,1,100,fashioncustom,https://www.reddit.com/user/fashioncustom,https://www.reddit.com/r/Customjerseyspro/comm...,Looking to customize a black volleyball jersey...,https://external-preview.redd.it/EBGZJNTa6m5jk...


# **Model Development**